# SASRec – Build Sequences & Train (Kaggle)

## Input
- **`nowplaying_processed.csv`** (đã merge với Spotify data, đã loại trùng user+time+track)
- Cột quan trọng: `user_id`, `timestamp`, `track_id` (Spotify ID)

## Pipeline
1. Load data (chỉ đọc 3 cột cần thiết để tiết kiệm RAM)
2. **K-core filtering**: Lọc user/track ít tương tác (lặp đến hội tụ)
3. Sắp xếp theo thời gian, build user sequences, loại bỏ consecutive duplicates
4. Encode item IDs, train/val/test split (leave-one-out)
5. Train SASRec model
6. Evaluate: **P@K, R@K, F1@K, Hit@K, NDCG@K, MRR** (K ∈ {5, 10, 20, 50})
7. Export model để deploy


In [ ]:
# ========================================
# CELL 1 – Install dependencies
# ========================================
!pip install torch pandas numpy tqdm scikit-learn matplotlib -q

In [ ]:
# ========================================
# CELL 2 – Imports & GPU check
# ========================================
import os
import json
import pickle
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from copy import deepcopy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

In [ ]:
# ========================================
# CELL 3 – Hyperparameters
# ========================================
CONFIG = {
    # Data filtering (k-core)
    'min_user_interactions': 10,   # Loại user nghe < 10 bài
    'min_track_interactions': 5,   # Loại track được nghe < 5 lần

    # Sequence
    'max_seq_len': 50,
    'min_seq_len': 5,              # User phải có >= 5+2=7 bài trong sequence

    # Model architecture
    'hidden_size': 128,
    'num_blocks': 2,
    'num_heads': 2,
    'dropout': 0.2,

    # Training
    'epochs': 50,
    'batch_size': 256,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'num_neg_samples': 100,
    'patience': 10,

    # Evaluation – K values
    'eval_k': [5, 10, 20, 50],
    'num_eval_neg': 100,

    'seed': 42
}

print('Hyperparameters:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
# ========================================
# CELL 4 – Load dataset
# ========================================
def find_file(pattern, search_dir='/kaggle/input'):
    for root, dirs, files in os.walk(search_dir):
        for f in files:
            if pattern.lower() in f.lower():
                return os.path.join(root, f)
    return None

data_path = (
    find_file('nowplaying_processed')
    or find_file('nowplaying_spotify')
    or 'nowplaying_processed.csv'
)
print(f'Dataset: {data_path}')

# Xem file có những cột nào
sample_df = pd.read_csv(data_path, nrows=1)
all_cols = sample_df.columns.tolist()
print(f'All columns: {all_cols}')

# Tự động tìm cột track ID
TRACK_ID_COL = next((c for c in ['track_id', 'spotify_track_id'] if c in all_cols), None)
assert TRACK_ID_COL, f'Không tìm thấy cột track_id! Cột hiện có: {all_cols}'

# CHỈ ĐỌC 3 CỘT CẦN THIẾT để tiết kiệm RAM tối đa
usecols = ['user_id', TRACK_ID_COL]
if 'timestamp' in all_cols:
    usecols.append('timestamp')

df = pd.read_csv(data_path, usecols=usecols)
if TRACK_ID_COL != 'spotify_track_id':
    df = df.rename(columns={TRACK_ID_COL: 'spotify_track_id'})

df = df.dropna()
print(f'\n📊 Thống kê TRƯỚC khi lọc:')
print(f'  Rows: {len(df):,}')
print(f'  Users:  {df["user_id"].nunique():,}')
print(f'  Tracks: {df["spotify_track_id"].nunique():,}')

In [ ]:
# ========================================
# CELL 5 – K-core Filtering (Lọc user và track ít tương tác)
# ========================================

MIN_USER = CONFIG['min_user_interactions']
MIN_TRACK = CONFIG['min_track_interactions']

print(f'🔍 K-core Filtering: min_user={MIN_USER}, min_track={MIN_TRACK}')
print(f'Trước lọc: {len(df):,} rows | {df["user_id"].nunique():,} users | {df["spotify_track_id"].nunique():,} tracks')

iteration = 0
while True:
    iteration += 1
    prev_len = len(df)

    # Bước 1: Lọc track ít tương tác
    track_counts = df['spotify_track_id'].value_counts()
    valid_tracks = track_counts[track_counts >= MIN_TRACK].index
    df = df[df['spotify_track_id'].isin(valid_tracks)]

    # Bước 2: Lọc user ít tương tác
    user_counts = df['user_id'].value_counts()
    valid_users = user_counts[user_counts >= MIN_USER].index
    df = df[df['user_id'].isin(valid_users)]

    print(f'  Iter {iteration}: {len(df):,} rows | {df["user_id"].nunique():,} users | {df["spotify_track_id"].nunique():,} tracks')

    # Dừng khi không còn gì bị loại (hội tụ)
    if len(df) == prev_len:
        print(f'✅ K-core hội tụ sau {iteration} vòng lặp!')
        break

print(f'\n📊 Thống kê SAU khi lọc:')
print(f'  Rows: {len(df):,}')
print(f'  Users:  {df["user_id"].nunique():,}')
print(f'  Tracks: {df["spotify_track_id"].nunique():,}')

In [ ]:
# ========================================
# CELL 6 – Build user sequences
# ========================================
print('Building user sequences...')

if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df = df.sort_values(['user_id', 'timestamp'])
    print('✅ Sorted by timestamp (Thứ tự thời gian chuẩn xác)')
else:
    df = df.sort_values('user_id')
    print('⚠️ Không có timestamp, giữ thứ tự gốc')

# item2id mapping (1-indexed, 0 = padding)
unique_items = df['spotify_track_id'].unique().tolist()
item2id = {item: idx + 1 for idx, item in enumerate(unique_items)}
id2item = {idx + 1: item for idx, item in enumerate(unique_items)}
num_items = len(unique_items)
print(f'Unique items: {num_items:,}')

df['item_id'] = df['spotify_track_id'].map(item2id)
user_sequences = df.groupby('user_id')['item_id'].apply(list).to_dict()

# ── LOẠI BỎ DUPLICATE LIÊN TIẾP ──
def dedup_consecutive(seq):
    if not seq: return []
    res = [seq[0]]
    for item in seq[1:]:
        if item != res[-1]:
            res.append(item)
    return res

user_sequences = {u: dedup_consecutive(s) for u, s in user_sequences.items()}
print('✅ Đã loại bỏ duplicate liên tiếp')

# Lọc sequence quá ngắn (< min_seq_len + 2 vì cần 2 bài cho val+test)
min_len = CONFIG['min_seq_len'] + 2
user_sequences = {u: s for u, s in user_sequences.items() if len(s) >= min_len}
print(f'Users sau filter sequence (>= {min_len} items): {len(user_sequences):,}')

seq_lengths = [len(s) for s in user_sequences.values()]
print(f'Seq length – mean: {np.mean(seq_lengths):.1f}, median: {np.median(seq_lengths):.0f}, max: {max(seq_lengths)}')

In [ ]:
# ========================================
# CELL 7 – Train/Val/Test split (Leave-One-Out)
# ========================================
train_data = {}
val_data   = {}
test_data  = {}

for user, sequence in user_sequences.items():
    train_data[user] = sequence[:-2]
    val_data[user]   = (sequence[:-2], sequence[-2])
    test_data[user]  = (sequence[:-1], sequence[-1])

print(f'Train users: {len(train_data):,}')
print(f'Val users:   {len(val_data):,}')
print(f'Test users:  {len(test_data):,}')
print(f'Vocab size:  {num_items + 1:,} (incl. padding=0)')

In [ ]:
# ========================================
# CELL 8 – Dataset & DataLoader
# ========================================
class SASRecDataset(Dataset):
    def __init__(self, user_sequences, num_items, max_seq_len, num_neg=100):
        self.sequences   = list(user_sequences.values())
        self.num_items   = num_items
        self.max_seq_len = max_seq_len
        self.num_neg     = num_neg

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        input_seq  = seq[:-1][-self.max_seq_len:]
        target_seq = seq[1:][-self.max_seq_len:]
        pad_len = self.max_seq_len - len(input_seq)
        input_seq  = [0] * pad_len + input_seq
        target_seq = [0] * pad_len + target_seq
        pos_set = set(seq)
        neg_items = []
        for _ in range(self.max_seq_len):
            n = random.randint(1, self.num_items)
            while n in pos_set:
                n = random.randint(1, self.num_items)
            neg_items.append(n)
        return (
            torch.LongTensor(input_seq),
            torch.LongTensor(target_seq),
            torch.LongTensor(neg_items)
        )

train_dataset = SASRecDataset(
    train_data, num_items=num_items,
    max_seq_len=CONFIG['max_seq_len'],
    num_neg=CONFIG['num_neg_samples']
)
train_loader = DataLoader(
    train_dataset, batch_size=CONFIG['batch_size'],
    shuffle=True, num_workers=0,
    pin_memory=(device.type == 'cuda')
)
print(f'Training batches: {len(train_loader):,}')

In [ ]:
# ========================================
# CELL 9 – SASRec Model Architecture
# ========================================
class PointWiseFeedForward(nn.Module):
    def __init__(self, hidden_size, dropout):
        super().__init__()
        self.fc1     = nn.Linear(hidden_size, hidden_size * 4)
        self.fc2     = nn.Linear(hidden_size * 4, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.gelu    = nn.GELU()
    def forward(self, x):
        return self.fc2(self.dropout(self.gelu(self.fc1(x))))


class SASRecBlock(nn.Module):
    def __init__(self, hidden_size, num_heads, dropout):
        super().__init__()
        self.attention = nn.MultiheadAttention(hidden_size, num_heads, dropout=dropout, batch_first=True)
        self.ffn   = PointWiseFeedForward(hidden_size, dropout)
        self.norm1 = nn.LayerNorm(hidden_size)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None, key_padding_mask=None):
        r = x; x = self.norm1(x)
        out, _ = self.attention(x, x, x, attn_mask=attn_mask,
                                key_padding_mask=key_padding_mask, need_weights=False)
        x = r + self.drop(out)
        r = x; x = self.norm2(x)
        return r + self.drop(self.ffn(x))


class SASRec(nn.Module):
    def __init__(self, num_items, hidden_size, num_blocks, num_heads, max_seq_len, dropout):
        super().__init__()
        self.item_emb   = nn.Embedding(num_items + 1, hidden_size, padding_idx=0)
        self.pos_emb    = nn.Embedding(max_seq_len + 1, hidden_size)
        self.dropout    = nn.Dropout(dropout)
        self.emb_norm   = nn.LayerNorm(hidden_size)
        self.blocks     = nn.ModuleList([SASRecBlock(hidden_size, num_heads, dropout) for _ in range(num_blocks)])
        self.final_norm = nn.LayerNorm(hidden_size)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def get_sequence_embedding(self, input_ids):
        B, L = input_ids.shape
        item_embs = self.item_emb(input_ids)
        positions = torch.arange(1, L + 1, device=input_ids.device).unsqueeze(0)
        x = self.emb_norm(self.dropout(item_embs + self.pos_emb(positions)))
        causal_mask = torch.triu(
            torch.ones(L, L, device=input_ids.device) * float('-inf'), diagonal=1
        )
        key_padding_mask = (input_ids == 0)
        for block in self.blocks:
            x = block(x, attn_mask=causal_mask, key_padding_mask=key_padding_mask)
        return self.final_norm(x)

    def forward(self, input_ids, pos_items, neg_items):
        seq_emb = self.get_sequence_embedding(input_ids)
        pos_emb = self.item_emb(pos_items)
        neg_emb = self.item_emb(neg_items)
        pos_scores = (seq_emb * pos_emb).sum(-1)
        neg_scores = (seq_emb * neg_emb).sum(-1)
        return pos_scores, neg_scores

    def predict(self, input_ids, candidate_items):
        seq_emb = self.get_sequence_embedding(input_ids)[:, -1, :]
        if candidate_items.dim() == 1:
            return seq_emb @ self.item_emb(candidate_items).T
        return (seq_emb.unsqueeze(1) * self.item_emb(candidate_items)).sum(-1)


model = SASRec(
    num_items=num_items,
    hidden_size=CONFIG['hidden_size'],
    num_blocks=CONFIG['num_blocks'],
    num_heads=CONFIG['num_heads'],
    max_seq_len=CONFIG['max_seq_len'],
    dropout=CONFIG['dropout']
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')

In [ ]:
# ========================================
# CELL 10 – Loss & Optimizer
# ========================================
def bce_loss(pos_scores, neg_scores, pos_items):
    mask     = (pos_items != 0).float()
    pos_loss = -F.logsigmoid(pos_scores) * mask
    neg_loss = -F.logsigmoid(-neg_scores) * mask
    return (pos_loss + neg_loss).sum() / mask.sum()

optimizer = torch.optim.Adam(
    model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay']
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
print('Optimizer & Scheduler sẵn sàng!')

In [ ]:
# ========================================
# CELL 11 – Evaluation function (sampled – dùng trong training loop)
# ========================================
@torch.no_grad()
def evaluate_sampled(model, eval_data, num_items, config, device, max_users=500):
    model.eval()
    ks = [k for k in config['eval_k'] if k <= 10]
    hit_counts  = {k: 0 for k in ks}
    ndcg_scores = {k: 0.0 for k in ks}
    mrr_score = 0.0
    total = 0

    users = list(eval_data.keys())[:max_users]
    for user in tqdm(users, desc='Val', leave=False):
        seq, target = eval_data[user]
        input_seq = seq[-config['max_seq_len']:]
        pad_len   = config['max_seq_len'] - len(input_seq)
        input_seq = [0] * pad_len + input_seq
        input_tensor = torch.LongTensor([input_seq]).to(device)

        pos_set = set(seq + [target])
        neg_items = []
        while len(neg_items) < config['num_eval_neg']:
            n = random.randint(1, num_items)
            if n not in pos_set:
                neg_items.append(n)

        candidates = torch.LongTensor([target] + neg_items).to(device)
        scores = model.predict(input_tensor, candidates)[0].cpu().numpy()
        rank = int(np.sum(scores > scores[0])) + 1

        for k in ks:
            if rank <= k:
                hit_counts[k] += 1
                ndcg_scores[k] += 1.0 / np.log2(rank + 1)
        mrr_score += 1.0 / rank
        total += 1

    metrics = {}
    for k in ks:
        metrics[f'Hit@{k}']  = hit_counts[k] / total
        metrics[f'NDCG@{k}'] = ndcg_scores[k] / total
    metrics['MRR'] = mrr_score / total
    return metrics

print('Evaluation function sẵn sàng!')

In [ ]:
# ========================================
# CELL 12 – Training loop
# ========================================
best_ndcg = 0.0
best_model_state = None
patience_counter = 0
history = {'train_loss': [], 'val_metrics': []}

print(f'\n🚀 Bắt đầu training SASRec trên {device}...\n')

for epoch in range(1, CONFIG['epochs'] + 1):
    model.train()
    total_loss, num_batches = 0.0, 0

    for input_ids, pos_items, neg_items in tqdm(train_loader, desc=f'Epoch {epoch}', leave=False):
        input_ids = input_ids.to(device)
        pos_items = pos_items.to(device)
        neg_items = neg_items.to(device)
        optimizer.zero_grad()
        pos_scores, neg_scores = model(input_ids, pos_items, neg_items)
        loss = bce_loss(pos_scores, neg_scores, pos_items)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()
        num_batches += 1

    scheduler.step()
    avg_loss = total_loss / num_batches
    history['train_loss'].append(avg_loss)

    if epoch % 5 == 0 or epoch == 1:
        val_metrics = evaluate_sampled(model, val_data, num_items, CONFIG, device)
        history['val_metrics'].append({'epoch': epoch, **val_metrics})
        ndcg10 = val_metrics.get('NDCG@10', 0)
        hit10  = val_metrics.get('Hit@10', 0)
        print(f'Epoch {epoch:3d} | Loss: {avg_loss:.4f} | '
              f'Val Hit@10: {hit10:.4f} | Val NDCG@10: {ndcg10:.4f} | '
              f'LR: {scheduler.get_last_lr()[0]:.6f}')
        if ndcg10 > best_ndcg:
            best_ndcg = ndcg10
            best_model_state = deepcopy(model.state_dict())
            patience_counter = 0
            print(f'  ✅ Best model! NDCG@10 = {best_ndcg:.4f}')
        else:
            patience_counter += 1
            if patience_counter >= CONFIG['patience']:
                print(f'\n⏹️ Early stopping tại epoch {epoch}')
                break

print(f'\n✅ Training xong! Best Val NDCG@10: {best_ndcg:.4f}')

In [ ]:
# ========================================
# CELL 13 – Learning curves
# ========================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], color='steelblue', linewidth=2)
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=0.3)

if history['val_metrics']:
    ep = [m['epoch'] for m in history['val_metrics']]
    axes[1].plot(ep, [m.get('Hit@10', 0) for m in history['val_metrics']], color='coral', marker='o')
    axes[1].set_title('Val Hit@10'); axes[1].grid(alpha=0.3)
    axes[2].plot(ep, [m.get('NDCG@10', 0) for m in history['val_metrics']], color='forestgreen', marker='o')
    axes[2].set_title('Val NDCG@10'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

In [ ]:
# ========================================
# CELL 14 – Đánh giá trên Test set (Full ranking)
# Metrics: P@K, R@K, F1@K, Hit@K, NDCG@K, MRR
# ========================================
model.load_state_dict(best_model_state)
model.eval()

@torch.no_grad()
def evaluate_full(model, eval_data, num_items, config, device):
    EVAL_KS = sorted(set(config['eval_k']))
    hit_counts  = {k: 0   for k in EVAL_KS}
    ndcg_scores = {k: 0.0 for k in EVAL_KS}
    precision   = {k: []  for k in EVAL_KS}
    recall      = {k: []  for k in EVAL_KS}
    f1          = {k: []  for k in EVAL_KS}
    mrr_score   = 0.0
    total       = 0

    all_items = torch.arange(1, num_items + 1, device=device)

    for user in tqdm(eval_data.keys(), desc='Test (full ranking)'):
        seq, target = eval_data[user]
        input_seq = seq[-config['max_seq_len']:]
        pad_len   = config['max_seq_len'] - len(input_seq)
        input_seq = [0] * pad_len + input_seq
        inp = torch.LongTensor([input_seq]).to(device)

        scores = model.predict(inp, all_items)[0].cpu().numpy()

        # Mask history (không mask target nếu nó cũng nằm trong history)
        input_set = set(input_seq)
        input_set.discard(0)
        input_set.discard(target)
        for idx in range(len(scores)):
            if (idx + 1) in input_set:
                scores[idx] = -1e9

        target_score = scores[target - 1]
        rank = int(np.sum(scores > target_score)) + 1

        mrr_score += 1.0 / rank
        for k in EVAL_KS:
            hit = 1 if rank <= k else 0
            if hit:
                hit_counts[k]  += 1
                ndcg_scores[k] += 1.0 / np.log2(rank + 1)
            p = hit / k
            r = float(hit)
            f = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
            precision[k].append(p)
            recall[k].append(r)
            f1[k].append(f)
        total += 1

    metrics = {}
    for k in EVAL_KS:
        metrics[f'Hit@{k}']  = hit_counts[k] / total
        metrics[f'NDCG@{k}'] = ndcg_scores[k] / total
        metrics[f'P@{k}']    = float(np.mean(precision[k]))
        metrics[f'R@{k}']    = float(np.mean(recall[k]))
        metrics[f'F1@{k}']   = float(np.mean(f1[k]))
    metrics['MRR'] = mrr_score / total
    return metrics

print('Đánh giá SASRec trên tập TEST (full ranking)...')
test_metrics = evaluate_full(model, test_data, num_items, CONFIG, device)

EVAL_KS = sorted(set(CONFIG['eval_k']))
print('\n' + '='*70)
print('📊 KẾT QUẢ SASRec TRÊN TẬP TEST (NowPlaying – Leave-One-Out)')
print('='*70)

header = f"  {'Metric':<10}"
for k in EVAL_KS:
    header += f" |  K={k:<4}"
print(header)
print('  ' + '-' * (len(header) - 2))

for prefix, label in [('P', 'P@K'), ('R', 'R@K'), ('F1', 'F1@K'), ('Hit', 'Hit@K'), ('NDCG', 'NDCG@K')]:
    row = f"  {label:<10}"
    for k in EVAL_KS:
        val = test_metrics.get(f'{prefix}@{k}', 0)
        row += f" | {val:.4f}  "
    print(row)
print(f"  {'MRR':<10} | {test_metrics['MRR']:.4f}")
print('='*70)

with open('test_metrics.json', 'w') as f:
    json.dump(test_metrics, f, indent=2)
print('\n✅ Saved test_metrics.json')

In [ ]:
# ========================================
# CELL 15 – Visualize evaluation metrics
# ========================================
import matplotlib.pyplot as plt
EVAL_KS = sorted(set(CONFIG['eval_k']))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('SASRec – Kết quả đánh giá trên Test Set (NowPlaying)',
             fontsize=14, fontweight='bold')

metric_groups = [
    ('Precision (P@K)',  'P',  '#2196F3'),
    ('Recall (R@K)',     'R',  '#4CAF50'),
    ('F1 Score (F1@K)', 'F1', '#FF9800'),
]

for ax, (title, prefix, color) in zip(axes, metric_groups):
    vals = [test_metrics.get(f'{prefix}@{k}', 0) for k in EVAL_KS]
    bars = ax.bar([f'K={k}' for k in EVAL_KS], vals,
                  color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.0002,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Score')
    ax.set_ylim(0, max(vals + [0.001]) * 1.35)
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('sasrec_eval_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved sasrec_eval_metrics.png')

In [ ]:
# ========================================
# CELL 16 – Export model và artifacts
# ========================================
import shutil

EXPORT_DIR = 'sasrec_export'
os.makedirs(EXPORT_DIR, exist_ok=True)

torch.save(best_model_state, os.path.join(EXPORT_DIR, 'sasrec_model.pt'))
print('✅ sasrec_model.pt')

export_config = {**CONFIG, 'num_items': num_items, 'test_metrics': test_metrics}
with open(os.path.join(EXPORT_DIR, 'sasrec_config.json'), 'w') as f:
    json.dump(export_config, f, indent=2)
print('✅ sasrec_config.json')

with open(os.path.join(EXPORT_DIR, 'item2id.pkl'), 'wb') as f:
    pickle.dump(item2id, f)
print(f'✅ item2id.pkl ({len(item2id):,} items)')

with open(os.path.join(EXPORT_DIR, 'id2item.pkl'), 'wb') as f:
    pickle.dump(id2item, f)
print('✅ id2item.pkl')

with open(os.path.join(EXPORT_DIR, 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)
print('✅ training_history.json')

for png in ['training_curves.png', 'sasrec_eval_metrics.png']:
    if os.path.exists(png):
        shutil.copy(png, EXPORT_DIR)

shutil.make_archive('sasrec_export', 'zip', EXPORT_DIR)
zip_size = os.path.getsize('sasrec_export.zip') / 1024 / 1024
print(f'\n📦 sasrec_export.zip ({zip_size:.1f} MB)')
print('\n📌 Kaggle: Download sasrec_export.zip từ Output panel')
print()
print('Nội dung thư mục export:')
for fname in sorted(os.listdir(EXPORT_DIR)):
    size = os.path.getsize(os.path.join(EXPORT_DIR, fname)) / 1024 / 1024
    print(f'  {fname} ({size:.1f} MB)')
print()
print('='*60)
print('✅ HOÀN THÀNH!')
print('Sau khi download sasrec_export.zip:')
print('  1. Giải nén vào model/session_base/sasrec/')
print('  2. Khởi động FastAPI: uvicorn main:app --reload')
print('  3. Test: POST /recommend/session/sasrec/')
print('='*60)